In [1]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

model_name = "BTCUSDT-5m"
src_path = folder_path / "clean" / f"{model_name}-v7-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,open,high,low,close,vol,label,train_mask,close_fracdiff
0,1704097200000,42514.218750,42537.531250,42510.839844,42510.839844,35.566738,-1,1,4836.836203
1,1704097500000,42510.839844,42510.851562,42477.601562,42482.000000,34.461250,0,0,4806.181054
2,1704097800000,42482.000000,42504.789062,42461.101562,42461.101562,36.952042,1,1,4793.580277
3,1704098100000,42461.101562,42490.320312,42461.101562,42472.511719,21.655741,1,1,4814.092371
4,1704098400000,42472.511719,42493.679688,42472.500000,42492.910156,30.919910,1,1,4833.413878


In [2]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(228573, 9)
Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'label',
       'train_mask', 'close_fracdiff'],
      dtype='object')


,timestamp,open,high,low,close,vol,label,train_mask,close_fracdiff
0,1704097200000,42514.218750,42537.531250,42510.839844,42510.839844,35.566738,-1,1,4836.836203
1,1704097500000,42510.839844,42510.851562,42477.601562,42482.000000,34.461250,0,0,4806.181054
2,1704097800000,42482.000000,42504.789062,42461.101562,42461.101562,36.952042,1,1,4793.580277
3,1704098100000,42461.101562,42490.320312,42461.101562,42472.511719,21.655741,1,1,4814.092371
4,1704098400000,42472.511719,42493.679688,42472.500000,42492.910156,30.919910,1,1,4833.413878


In [3]:
# tactical drop extra timeframe and non-stationary data
# prepare for scaler : scaler is order strict

df.drop(columns=["open", "high", "low","close", "vol"], inplace=True)
print(df.shape)
df.head()

(228573, 4)


,timestamp,label,train_mask,close_fracdiff
0,1704097200000,-1,1,4836.836203
1,1704097500000,0,0,4806.181054
2,1704097800000,1,1,4793.580277
3,1704098100000,1,1,4814.092371
4,1704098400000,1,1,4833.413878


# At this point

- scale all cols - pick training case later

In [4]:
# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)

n = len(df)

# Chronological split on ALL rows (includes timeout rows — preserves time boundaries)
train_df = df.iloc[:int(n*0.8)]
val_df   = df.iloc[int(n*0.8):int(n*0.9)]
test_df  = df.iloc[int(n*0.9):]

# print(f"All rows  — Train: {len(train_all):,} | Val: {len(val_all):,} | Test: {len(test_all):,}")
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


Trainable — Train: 182,858 | Val: 22,857 | Test: 22,858


In [5]:
# check


# However — there's a catch for your case. You're using a chronological split

In [6]:
print(df.columns)
print(len(df.columns))
del df

Index(['timestamp', 'label', 'train_mask', 'close_fracdiff'], dtype='object')
4


In [7]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

# scale all feature columns — pick subset at training time
scale_cols = ['close_fracdiff']

# not scaled: timestamp, ts_45m, label, train_mask,
#             15STR_confirmed, 45STR_confirmed  (categorical: -1/0/1)


scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
val_df[scale_cols]   = scaler.transform(val_df[scale_cols])        # transform only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler-V6.pkl")
print("Scaler saved.")
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")


Scaler saved.
Train: (182858, 4) | Val: (22857, 4) | Test: (22858, 4)


In [8]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train-v7.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val-v7.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test-v7.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSDT-5m-train-v7.jsonl
Saved val: 202603-BTCUSDT-5m-val-v7.jsonl
Saved test:  202603-BTCUSDT-5m-test-v7.jsonl


In [9]:
non_feature = ['timestamp', 'label', 'train_mask']
feature_cols = [c for c in train_df.columns if c not in non_feature]

print(f"Train columns  : {len(train_df.columns)}  → {list(train_df.columns)}")
print(f"Feature columns: {len(feature_cols)}  → {feature_cols}")
print(f"Scaled columns : {len(scale_cols)}  → {scale_cols}")


Train columns  : 4  → ['timestamp', 'label', 'train_mask', 'close_fracdiff']
Feature columns: 1  → ['close_fracdiff']
Scaled columns : 1  → ['close_fracdiff']


In [10]:
train_df.nunique()

timestamp         182858
label                  3
train_mask             2
close_fracdiff    182858
dtype: int64